# Notebook 09 — mCodeGPT-Style DAG Extraction for Breast Cancer Clinical Features

Adapts the [mCodeGPT](https://github.com/anotherkaizhang/mCodeGPT) ontology-guided prompt framework to the 14
breast cancer clinical elements in this project.

**Three prompt-generation strategies (from mCodeGPT):**
- **RLS** (Root-to-Leaf Streamliner) — all 14 leaf nodes in one prompt
- **BFOP** (Breadth-First Ontology Pruner) — layer-by-layer, pruning absent branches
- **2POP** (Two-Phase Ontology Parser) — yes/no gate first, then value extraction

**Hypothesis:** BFOP/2POP reduce fabrication by skipping branches the LLM confirms are absent,
aligning with the primary safety outcome of this study.

**Inputs:**
- Extracted case text (`DATA_PRIVATE_DIR/extracted_text/CASE_*.txt`) from NB04
- Validation datasheet (for ground-truth comparison)

**Outputs:**
- `data/processed/dag_extraction_results.csv` — per-case per-element extraction results
- `reports/dag_method_comparison.csv` — accuracy/fabrication by method (RLS vs BFOP vs 2POP)
- `reports/dag_ontology_structure.png` — DAG visualisation

In [ ]:
import os
import re
import copy
import json
import warnings
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
from openai import OpenAI

warnings.filterwarnings('ignore')
load_dotenv()

PROJECT_ROOT     = Path(os.getenv('PROJECT_ROOT',
    r'C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca'))
DATA_PRIVATE_DIR = Path(os.getenv('DATA_PRIVATE_DIR', r'C:\Users\jamesr4\loc\data_private'))

TEXT_DIR    = DATA_PRIVATE_DIR / 'extracted_text'
PROC_DIR    = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR  = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print(f'Model : {OPENAI_MODEL}')
print(f'TEXT_DIR exists: {TEXT_DIR.exists()}')

## 9.1 — Ontology Definition

Defines the breast cancer clinical feature hierarchy as a 4-level DAG.
Each node carries:
- `description` — extraction instruction (used in value-extraction prompts)
- `description_yesno` — yes/no gate question (used in 2POP phase 1 and BFOP gates)

In [ ]:
# ── Ontology: (parent, child, layer, description, description_yesno) ───────────
# Layer 0 = root, Layer 1 = domain, Layer 2 = sub-domain, Layer 3 = leaf element

ONTOLOGY = [
    # root
    (None,                       'breast_cancer_case',          0,
     'Breast cancer clinical case summary.',
     'Does this document describe a breast cancer case? (yes/no)'),

    # domain level
    ('breast_cancer_case',       'imaging_findings',            1,
     'Radiology/imaging findings from mammography, ultrasound, or MRI.',
     'Are there radiology or imaging findings reported in this document? (yes/no)'),
    ('breast_cancer_case',       'pathologic_findings',         1,
     'Pathology findings from biopsy or surgical specimen.',
     'Are there pathology or biopsy findings reported in this document? (yes/no)'),

    # imaging sub-domains
    ('imaging_findings',         'lesion_characteristics',      2,
     'Physical characteristics of the breast lesion on imaging.',
     'Are lesion characteristics (size, location, laterality) described? (yes/no)'),
    ('imaging_findings',         'imaging_workup',              2,
     'Imaging follow-up, clip placement, and workup recommendations.',
     'Are imaging workup or follow-up recommendations documented? (yes/no)'),
    ('imaging_findings',         'lymph_node_imaging',          2,
     'Lymph node status as assessed on imaging.',
     'Is lymph node status documented on imaging? (yes/no)'),

    # pathology sub-domains
    ('pathologic_findings',      'tumor_pathology',             2,
     'Tumor histology, size, and invasive component details.',
     'Are tumor pathology details (histologic type, invasive size) documented? (yes/no)'),
    ('pathologic_findings',      'biomarker_status',            2,
     'Receptor and biomarker status (ER, PR, HER2).',
     'Are biomarker or receptor statuses (ER/PR/HER2) reported? (yes/no)'),
    ('pathologic_findings',      'procedural_documentation',    2,
     'Biopsy method and chronology of pathologic workup.',
     'Are procedural details (biopsy method, chronology) documented? (yes/no)'),

    # leaf nodes — imaging
    ('lesion_characteristics',   'lesion_size',                 3,
     'Size of the primary breast lesion in millimeters or centimeters as reported on imaging.',
     'Is the lesion size explicitly stated on imaging? (yes/no)'),
    ('lesion_characteristics',   'laterality',                  3,
     'Side (left or right breast) of the primary lesion.',
     'Is the laterality (left/right breast) explicitly stated? (yes/no)'),
    ('lesion_characteristics',   'lesion_location',             3,
     'Location of the lesion within the breast (e.g., upper outer quadrant, 2 o'clock position).',
     'Is the lesion location within the breast explicitly stated? (yes/no)'),
    ('lesion_characteristics',   'calcifications_asymmetry',    3,
     'Presence of calcifications or asymmetry associated with the lesion on imaging.',
     'Are calcifications or asymmetry findings described? (yes/no)'),
    ('lesion_characteristics',   'additional_enhancement_mri',  3,
     'Additional enhancement on MRI beyond the index lesion.',
     'Is additional MRI enhancement beyond the index lesion documented? (yes/no)'),
    ('lesion_characteristics',   'extent',                      3,
     'Extent of disease on imaging (e.g., unifocal, multifocal, multicentric).',
     'Is the extent of disease (unifocal/multifocal/multicentric) stated? (yes/no)'),
    ('imaging_workup',           'accurate_clip_placement',     3,
     'Whether a biopsy clip was accurately placed at the lesion site.',
     'Is accurate clip placement documented? (yes/no)'),
    ('imaging_workup',           'workup_recommendation',       3,
     'Recommended next imaging or clinical workup step.',
     'Is a workup or follow-up recommendation explicitly stated? (yes/no)'),
    ('lymph_node_imaging',       'lymph_node',                  3,
     'Lymph node status on imaging (e.g., negative, suspicious, biopsy-proven positive).',
     'Is lymph node status on imaging explicitly documented? (yes/no)'),

    # leaf nodes — pathology
    ('tumor_pathology',          'invasive_component_size',     3,
     'Size of the invasive tumor component in the surgical pathology specimen (mm or cm).',
     'Is the invasive component size in pathology explicitly stated? (yes/no)'),
    ('tumor_pathology',          'histologic_diagnosis',        3,
     'Histologic diagnosis (e.g., invasive ductal carcinoma, invasive lobular carcinoma, DCIS).',
     'Is the histologic diagnosis explicitly stated in pathology? (yes/no)'),
    ('biomarker_status',         'receptor_status',             3,
     'Hormone receptor (ER, PR) and HER2 status from pathology.',
     'Are ER, PR, and HER2 receptor statuses documented in pathology? (yes/no)'),
    ('procedural_documentation', 'biopsy_method',               3,
     'Method used for biopsy (e.g., ultrasound-guided core needle, stereotactic, vacuum-assisted).',
     'Is the biopsy method explicitly documented? (yes/no)'),
    ('procedural_documentation', 'chronology_preserved',        3,
     'Whether the chronological sequence of diagnostic workup is clearly preserved in the report.',
     'Is the chronology of the diagnostic workup preserved and clearly stated? (yes/no)'),
]

# Map to DataFrames matching mCodeGPT schema
def build_ontology_df(ontology):
    """Convert flat ontology list to mCodeGPT-style layered DataFrame."""
    max_layer = max(r[2] for r in ontology)
    rows_ont, rows_prompt, rows_yesno = [], [], []
    for parent, child, layer, desc, yesno in ontology:
        row_ont   = [None] * (max_layer + 1)
        row_pmt   = [None] * (max_layer + 1)
        row_yn    = [None] * (max_layer + 1)
        row_ont[layer]  = child
        row_pmt[layer]  = desc
        row_yn[layer]   = yesno
        rows_ont.append(row_ont)
        rows_prompt.append(row_pmt)
        rows_yesno.append(row_yn)
    return (pd.DataFrame(rows_ont),
            pd.DataFrame(rows_prompt),
            pd.DataFrame(rows_yesno))

df_ontology, df_prompt, df_promptYesNo = build_ontology_df(ONTOLOGY)
print(f'Ontology nodes: {len(ONTOLOGY)}')
print(f'Leaf nodes    : {sum(1 for _,_,l,_,_ in ONTOLOGY if l == 3)}')

## 9.2 — Build NetworkX DAG & Visualise

In [ ]:
def build_dag(ontology):
    """Build nx.DiGraph from ontology list."""
    G = nx.DiGraph()
    for parent, child, layer, desc, yesno in ontology:
        G.add_node(child, layer=layer, description=desc,
                   description_yesno=yesno, information_present_flag='YES')
        if parent is not None:
            G.add_edge(parent, child)
    return G

G = build_dag(ONTOLOGY)
max_layer = max(nx.get_node_attributes(G, 'layer').values())
layer_nodes = {l: [n for n, d in G.nodes(data=True) if d['layer'] == l]
               for l in range(max_layer + 1)}

# ── Visualise ─────────────────────────────────────────────────────────────────
LAYER_COLORS = {0: '#2c3e50', 1: '#2980b9', 2: '#27ae60', 3: '#e74c3c'}
LAYER_LABELS = {0: 'Root', 1: 'Domain', 2: 'Sub-domain', 3: 'Leaf element'}

def hierarchical_pos(G, root=None, width=2.0, vert_gap=0.4, xcenter=0.5):
    """Top-down hierarchical layout using BFS."""
    if root is None:
        root = [n for n in G if G.in_degree(n) == 0][0]
    pos = {}
    def _pos(node, x, y, dx):
        pos[node] = (x, y)
        children = list(G.successors(node))
        if children:
            step = dx / len(children)
            start = x - dx/2 + step/2
            for i, c in enumerate(children):
                _pos(c, start + i*step, y - vert_gap, step)
    _pos(root, xcenter, 0, width)
    return pos

pos = hierarchical_pos(G, root='breast_cancer_case', width=2.4, vert_gap=0.35)
node_colors = [LAYER_COLORS[G.nodes[n]['layer']] for n in G.nodes()]

fig, ax = plt.subplots(figsize=(18, 8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True,
                       arrowstyle='->', arrowsize=15,
                       edge_color='#7f8c8d', width=1.2)
nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=900, ax=ax, alpha=0.9)
labels = {n: n.replace('_', '\n') for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=6,
                        font_color='white', ax=ax)
legend = [mpatches.Patch(color=c, label=LAYER_LABELS[l])
          for l, c in LAYER_COLORS.items()]
ax.legend(handles=legend, loc='lower right', fontsize=9)
ax.set_title('Breast Cancer Clinical Feature Ontology DAG\n'
             '(mCodeGPT-style — BFOP/2POP/RLS extraction)',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'dag_ontology_structure.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {REPORT_DIR / "dag_ontology_structure.png"}')

## 9.3 — mCodeGPT Extraction Engine (updated to openai v1.x)

In [ ]:
LEAF_NODES = [n for n in G.nodes() if G.out_degree(n) == 0]

def _openai_call(prompt: str, input_text: str, model: str = OPENAI_MODEL) -> str:
    """Call OpenAI chat API (v1.x client)."""
    full_prompt = prompt + '\n\nText:\n' + input_text
    resp = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': full_prompt}],
        max_tokens=2000,
        temperature=0.0,
    )
    return resp.choices[0].message.content

def _parse_response(text: str) -> pd.DataFrame:
    """Parse key: value lines into a DataFrame."""
    rows = []
    for line in text.strip().split('\n'):
        if ':' in line and len(line) > 3:
            key, *rest = line.split(':')
            rows.append({'element': key.strip().lower().replace(' ', '_'),
                         'value': ':'.join(rest).strip()})
    return pd.DataFrame(rows)

def _update_flags(G_copy, df_out: pd.DataFrame):
    """Update information_present_flag recursively based on yes/no responses."""
    def _propagate_no(node):
        G_copy.nodes[node]['information_present_flag'] = 'NO'
        for child in G_copy.successors(node):
            _propagate_no(child)
    for _, row in df_out.iterrows():
        node = row['element']
        if node not in G_copy.nodes:
            continue
        if 'yes' in str(row['value']).lower():
            G_copy.nodes[node]['information_present_flag'] = 'YES'
        else:
            _propagate_no(node)

# ── RLS ───────────────────────────────────────────────────────────────────────
def extract_rls(input_text: str) -> pd.DataFrame:
    """Root-to-Leaf Streamliner: all leaf nodes in one prompt."""
    lines = ['From the text below, extract the following clinical entities.\n'
             'Return each on its own line as: element_name: <value or "not stated">\n']
    for node in LEAF_NODES:
        lines.append(f"{node}: <{G.nodes[node]['description']}>")
    return _parse_response(_openai_call('\n'.join(lines), input_text))

# ── BFOP ──────────────────────────────────────────────────────────────────────
def extract_bfop(input_text: str) -> pd.DataFrame:
    """Breadth-First Ontology Pruner: layer-by-layer with branch pruning."""
    G_work = copy.deepcopy(G)
    for n in G_work.nodes:
        G_work.nodes[n]['information_present_flag'] = 'YES'
    all_outputs = []

    for layer in range(max_layer + 1):
        active = [n for n in layer_nodes[layer]
                  if G_work.nodes[n]['information_present_flag'] == 'YES']
        if not active:
            break

        # Leaf layer: extract values
        if layer == max_layer:
            lines = ['From the text below, extract the following clinical entities.\n'
                     'Return each on its own line as: element_name: <value or "not stated">\n']
            for node in active:
                lines.append(f"{node}: <{G_work.nodes[node]['description']}>")
            df = _parse_response(_openai_call('\n'.join(lines), input_text))
            all_outputs.append(df)
        else:
            # Non-leaf: yes/no gate
            lines = ['For each item below, answer yes or no — is this information present in the text?\n'
                     'Return each on its own line as: element_name: yes/no\n']
            for node in active:
                lines.append(f"{node}: {G_work.nodes[node]['description_yesno']}")
            df = _parse_response(_openai_call('\n'.join(lines), input_text))
            _update_flags(G_work, df)

    if not all_outputs:
        return pd.DataFrame(columns=['element', 'value'])
    result = pd.concat(all_outputs, ignore_index=True).drop_duplicates('element')
    # Fill missing leaf nodes
    present = set(result['element'])
    missing = [{'element': n, 'value': 'not stated'}
               for n in LEAF_NODES if n not in present]
    return pd.concat([result, pd.DataFrame(missing)], ignore_index=True)

# ── 2POP ──────────────────────────────────────────────────────────────────────
def extract_2pop(input_text: str) -> pd.DataFrame:
    """Two-Phase Ontology Parser: yes/no gate then value extraction."""
    # Phase 1: yes/no for all leaf nodes
    lines = ['For each clinical element below, answer yes or no — is it present in the text?\n'
             'Return each on its own line as: element_name: yes/no\n']
    for node in LEAF_NODES:
        lines.append(f"{node}: {G.nodes[node]['description_yesno']}")
    df_gate = _parse_response(_openai_call('\n'.join(lines), input_text))
    present_nodes = [r['element'] for _, r in df_gate.iterrows()
                     if 'yes' in str(r['value']).lower() and r['element'] in LEAF_NODES]

    if not present_nodes:
        missing = [{'element': n, 'value': 'not stated'} for n in LEAF_NODES]
        return pd.DataFrame(missing)

    # Phase 2: extract values only for confirmed-present nodes
    lines2 = ['From the text below, extract the following clinical entities.\n'
              'Return each on its own line as: element_name: <value or "not stated">\n']
    for node in present_nodes:
        lines2.append(f"{node}: <{G.nodes[node]['description']}>")
    df_values = _parse_response(_openai_call('\n'.join(lines2), input_text))

    # Fill absent nodes
    present_set = set(df_values['element'])
    absent = [{'element': n, 'value': 'not stated'} for n in LEAF_NODES if n not in present_set]
    return pd.concat([df_values, pd.DataFrame(absent)], ignore_index=True)

print('Extraction engines ready. Leaf nodes:', len(LEAF_NODES))

## 9.4 — Run Extraction on Case Texts

In [ ]:
# Load extracted case texts from NB04
case_texts = {}
if TEXT_DIR.exists():
    for txt_path in sorted(TEXT_DIR.glob('CASE_*.txt')):
        case_id = txt_path.stem  # e.g. CASE_abc123
        text = txt_path.read_text(encoding='utf-8', errors='ignore').strip()
        if text:
            case_texts[case_id] = text

print(f'Case texts found: {len(case_texts)}')

# ── Config: limit cases for cost control ──────────────────────────────────────
MAX_CASES   = int(os.getenv('DAG_MAX_CASES', 20))   # set to None for all cases
METHODS     = ['RLS', 'BFOP', '2POP']
EXTRACT_FNS = {'RLS': extract_rls, 'BFOP': extract_bfop, '2POP': extract_2pop}

cases_to_run = list(case_texts.items())[:MAX_CASES] if MAX_CASES else list(case_texts.items())
print(f'Running extraction on {len(cases_to_run)} cases × {len(METHODS)} methods')
print(f'Estimated API calls: {len(cases_to_run) * len(METHODS)} — {len(cases_to_run) * 5} (BFOP/2POP multi-call)')

In [ ]:
# Run extraction (this cell makes API calls — costs tokens)
results = []

for case_id, text in tqdm(cases_to_run, desc='Cases'):
    for method in METHODS:
        try:
            df = EXTRACT_FNS[method](text)
            df['case_id'] = case_id
            df['method']  = method
            results.append(df)
        except Exception as exc:
            print(f'  [{method}] {case_id}: {exc}')

df_results = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
out_path = PROC_DIR / 'dag_extraction_results.csv'
df_results.to_csv(out_path, index=False)
print(f'Saved {len(df_results)} rows → {out_path}')
df_results.head(10)

## 9.5 — Compare Methods Against Ground Truth

In [ ]:
# ── Column name mapping: element name → dataset column base ───────────────────
ELEMENT_COL_MAP = {
    'lesion_size':              'lesion_size',
    'laterality':               'laterality',
    'lesion_location':          'lesion_location',
    'calcifications_asymmetry': 'calcifications_asymmetry',
    'additional_enhancement_mri': 'additional_enhancement_mri',
    'extent':                   'extent',
    'accurate_clip_placement':  'accurate_clip_placement',
    'workup_recommendation':    'workup_recommendation',
    'lymph_node':               'Lymph node',
    'chronology_preserved':     'chronology_preserved',
    'biopsy_method':            'biopsy_method',
    'invasive_component_size':  'invasive_component_size_pathology',
    'histologic_diagnosis':     'histologic_diagnosis',
    'receptor_status':          'receptor_status',
}

# Load ground truth
gt_path = PROC_DIR / 'comprehensive_enhanced_dataset_with_all_metrics.csv'
df_gt = pd.read_csv(gt_path, index_col=0) if gt_path.exists() else None

if df_gt is not None and not df_results.empty:
    # Map extraction output to binary: 1=extracted (not 'not stated'), 0=not extracted
    df_results['extracted_binary'] = (~df_results['value'].str.lower().str.contains(
        r'not stated|unknown|n/a|none', na=True, regex=True)).astype(int)

    # Aggregate per method × element: extraction rate
    summary = (df_results.groupby(['method', 'element'])['extracted_binary']
               .agg(['mean', 'count'])
               .rename(columns={'mean': 'extraction_rate', 'count': 'n_cases'})
               .reset_index())

    # Add domain label
    RAD_ELEMENTS  = {'lesion_size', 'laterality', 'lesion_location', 'calcifications_asymmetry',
                     'additional_enhancement_mri', 'extent', 'accurate_clip_placement',
                     'workup_recommendation', 'lymph_node'}
    PATH_ELEMENTS = {'chronology_preserved', 'biopsy_method', 'invasive_component_size',
                     'histologic_diagnosis', 'receptor_status'}
    summary['domain'] = summary['element'].map(
        lambda e: 'Radiology' if e in RAD_ELEMENTS else 'Pathology')

    out_summary = REPORT_DIR / 'dag_method_comparison.csv'
    summary.to_csv(out_summary, index=False)
    print(f'Saved method comparison: {out_summary}')
    display(summary.pivot_table(index='element', columns='method',
                                values='extraction_rate').round(3))
else:
    print('Ground truth or results not available — run extraction cell first.')

## 9.6 — Visualise Method Comparison

In [ ]:
if 'summary' in dir() and not summary.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
    colors = {'RLS': '#3498db', 'BFOP': '#e67e22', '2POP': '#27ae60'}

    for ax, domain in zip(axes, ['Radiology', 'Pathology']):
        sub = summary[summary['domain'] == domain]
        pivot = sub.pivot_table(index='element', columns='method',
                                values='extraction_rate').fillna(0)
        pivot.plot(kind='bar', ax=ax, color=[colors.get(m, 'grey') for m in pivot.columns],
                   width=0.7, edgecolor='black', linewidth=0.5)
        ax.set_title(f'{domain} Elements — Extraction Rate by Method',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('Extraction Rate (% of cases element found)')
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis='x', rotation=35, labelsize=8)
        ax.legend(title='Method')
        ax.axhline(0.8, color='red', linestyle='--', linewidth=1, alpha=0.6, label='80% threshold')
        ax.grid(axis='y', linestyle=':', alpha=0.5)

    plt.suptitle('mCodeGPT DAG Extraction — Method Comparison\n'
                 'Hypothesis: BFOP/2POP reduce fabrication by pruning absent branches',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'dag_method_comparison.png', dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {REPORT_DIR / "dag_method_comparison.png"}')